# PCA Cost Analysis: Conservative vs E-Reserve Energy Models

This notebook performs Principal Component Analysis (PCA) on energy reserve cost data to understand the underlying patterns and relationships between different cost components across model types (envelope vs e-reserve).

## 1. Load Required Packages and Set Configuration

In [ ]:
# Load packages
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
import plotly.express as px
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import read_KPI_adequacy

# Set visualization dimensions
dim = (1000, 500)

## 2. Define Solution Scenarios and Load KPI Data

In [ ]:
ss = [
    {'solution_folder': f"RTS-GMLC_v32.2s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
]

gcd_KPI_adequacy, gcdi_KPI_adequacy = read_KPI_adequacy(ss)

## 3. Data Filtering and Preparation

In [ ]:
# Create filter for days with data for both model types
filter_ = gcd_KPI_adequacy.pivot(
    index='day',
    columns='model_type',
    values='solution_id'
).dropna().index

print(f"Number of valid days: {len(filter_)}")
print(f"Days range: {filter_.min()} to {filter_.max()}")

## 4. Cost Difference Analysis Between Model Types

In [ ]:
# Group by day and calculate differences between model types
cost_comparison = gcd_KPI_adequacy.pivot_table(
    index='day',
    columns='model_type',
    values='OV_uc',
    aggfunc='first'
).dropna()

# Calculate the difference: envelope - e-reserve
cost_comparison['OV_uc_diff'] = cost_comparison['envelope'] - cost_comparison['e-reserve']
cost_comparison['OV_uc_diff_pct'] = (cost_comparison['envelope'] - cost_comparison['e-reserve']) / cost_comparison['e-reserve'] * 100

print("Cost comparison summary:")
print(f"Mean difference: {cost_comparison['OV_uc_diff'].mean():.2f}")
print(f"Mean percentage difference: {cost_comparison['OV_uc_diff_pct'].mean():.2f}%")
cost_comparison.head()

## 5. Prepare Data for PCA Analysis

In [ ]:
# Get all cost columns for PCA analysis
cost_columns = [col for col in gcd_KPI_adequacy.columns if 'cost' in col and '_uc' in col]
print("Cost columns for PCA:", cost_columns)

# Create a comprehensive dataset with all cost data
pca_data = gcd_KPI_adequacy.pivot_table(
    index='day',
    columns='model_type', 
    values=['OV_uc'] + cost_columns,
    aggfunc='first'
).dropna()

# Flatten column names for easier handling
pca_data.columns = ['_'.join(col) for col in pca_data.columns]
print(f"PCA dataset shape: {pca_data.shape}")
pca_data.head()

## 6. Perform PCA Analysis

In [ ]:
# Prepare data for PCA
# Standardize the features
scaler = StandardScaler()
pca_data_scaled = scaler.fit_transform(pca_data)

# Perform PCA
pca = PCA()
pca_result = pca.fit_transform(pca_data_scaled)

# Create a DataFrame with PCA results
pca_df = pd.DataFrame(
    pca_result, 
    index=pca_data.index,
    columns=[f'PC{i+1}' for i in range(pca_result.shape[1])]
)

# Calculate explained variance ratio
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("PCA Results:")
print(f"Number of components: {len(explained_variance)}")
print(f"First 5 components explain {cumulative_variance[4]:.2%} of variance")
print(f"First 2 components explain {cumulative_variance[1]:.2%} of variance")

## 7. Visualize PCA Results - Explained Variance

In [ ]:
# Visualize PCA results
fig1 = px.bar(
    x=range(1, min(11, len(explained_variance) + 1)), 
    y=explained_variance[:10],
    title='PCA Explained Variance Ratio (First 10 Components)',
    labels={'x': 'Principal Component', 'y': 'Explained Variance Ratio'}
)
fig1.update_layout(width=dim[0], height=dim[1])
fig1.show()

# Scree plot with cumulative variance
fig2 = px.line(
    x=range(1, min(11, len(cumulative_variance) + 1)), 
    y=cumulative_variance[:10],
    title='Cumulative Explained Variance',
    labels={'x': 'Number of Components', 'y': 'Cumulative Explained Variance'}
)
fig2.update_layout(width=dim[0], height=dim[1])
fig2.show()

## 8. PCA Biplot with Feature Loadings

In [ ]:
# Biplot: PCA scatter plot with feature loadings
fig3 = px.scatter(
    pca_df.reset_index(), 
    x='PC1', 
    y='PC2',
    hover_data=['day'],
    title=f'PCA Biplot (PC1 vs PC2) - {cumulative_variance[1]:.1%} of variance explained',
    labels={
        'PC1': f'PC1 ({explained_variance[0]:.1%} variance)',
        'PC2': f'PC2 ({explained_variance[1]:.1%} variance)'
    }
)

# Add feature loadings as arrows (scaled for visibility)
loadings = pca.components_[:2].T * np.sqrt(pca.explained_variance_[:2])
scale_factor = 3  # Adjust this to make arrows more visible

for i, (feature, loading) in enumerate(zip(pca_data.columns, loadings)):
    fig3.add_annotation(
        ax=0, ay=0,
        x=loading[0] * scale_factor, 
        y=loading[1] * scale_factor,
        xref='x', yref='y',
        axref='x', ayref='y',
        showarrow=True,
        arrowhead=2,
        arrowcolor='red',
        arrowwidth=1,
        text=feature[:20] + ('...' if len(feature) > 20 else ''),  # Truncate long names
        font=dict(size=8)
    )

fig3.update_layout(width=dim[0], height=dim[1])
fig3.show()

## 9. Feature Importance Analysis

In [ ]:
# Feature importance analysis
# Get the loadings (coefficients) for the first two principal components
loadings_df = pd.DataFrame(
    pca.components_[:5].T,  # First 5 components
    columns=[f'PC{i+1}' for i in range(5)],
    index=pca_data.columns
)

# Calculate the magnitude of loadings for each feature
loadings_df['magnitude'] = np.sqrt((loadings_df**2).sum(axis=1))
loadings_df = loadings_df.sort_values('magnitude', ascending=False)

print("Top 10 most important features based on PCA loadings:")
print(loadings_df.head(10))

# Visualize feature loadings for PC1 and PC2
fig4 = px.scatter(
    loadings_df.reset_index(), 
    x='PC1', 
    y='PC2',
    hover_data=['index', 'magnitude'],
    title='Feature Loadings on PC1 vs PC2',
    labels={'index': 'Feature'}
)
fig4.update_layout(width=dim[0], height=dim[1])
fig4.show()

## 10. Correlation Analysis: Cost Differences vs Principal Components

In [ ]:
# Correlation analysis between OV_uc differences and PCA components
# Add the cost difference data to PCA results
pca_analysis = pca_df.copy()
pca_analysis['OV_uc_diff'] = cost_comparison['OV_uc_diff']
pca_analysis['OV_uc_diff_pct'] = cost_comparison['OV_uc_diff_pct']

# Calculate correlations between cost differences and principal components
correlations = pca_analysis[['PC1', 'PC2', 'PC3', 'PC4', 'PC5']].corrwith(pca_analysis['OV_uc_diff_pct'])
print("Correlations between OV_uc percentage differences and Principal Components:")
print(correlations.sort_values(key=abs, ascending=False))

# Scatter plot: PC1 vs cost difference
fig5 = px.scatter(
    pca_analysis.reset_index(),
    x='PC1',
    y='OV_uc_diff_pct',
    hover_data=['day'],
    title='PC1 vs OV_uc Cost Difference (%)',
    labels={
        'PC1': f'PC1 ({explained_variance[0]:.1%} variance)',
        'OV_uc_diff_pct': 'Cost Difference (Envelope - E-reserve) %'
    }
)
fig5.update_layout(width=dim[0], height=dim[1])
fig5.show()

## Summary and Insights

This notebook provides a comprehensive PCA analysis of energy reserve cost data comparing envelope and e-reserve model types. Key insights:

1. **Cost Differences**: Shows the systematic differences in total costs (OV_uc) between model types
2. **Principal Components**: Identifies the main patterns of variance in cost data across different cost categories
3. **Feature Importance**: Highlights which cost components contribute most to the overall variance
4. **Correlations**: Reveals relationships between cost differences and underlying cost patterns

The analysis helps understand which cost factors drive the differences between conservative envelope approaches and e-reserve strategies in energy system optimization.